# Laboratorium terbuka: pemantulan batu di permukaan air

Notebook ini merupakan pendamping komputasi mandiri untuk Bab 4. Seluruh contoh memakai Python terbuka dan dapat dijalankan secara luring setelah paket pada [requirements.lock](requirements.lock) dipasang.

Kita akan memeriksa lima akibat model bab: lintasan terbang bebas, stabilisasi giroskopik, gerak tumbukan batu persegi, syarat kecepatan kritis, dan penyusutan panjang lompatan. Semua nilai parameter di bawah dipilih untuk demonstrasi deterministik. **Tidak ada data eksperimen dalam penutupan sumber bab ini, sehingga hasil notebook bukan validasi terhadap eksperimen.**

**ID unit:** `O005-LEGA-V101-CH04`  
**ID notebook:** `O005-LEGA-V101-CH04-NB01`  
**Lisensi:** CC BY-NC-SA 4.0  
**Asal komponen:** pendamping komputasi baru berdasarkan persamaan Bab 4; tidak menggantikan teks atau latihan bab.

## 1. Parameter dan syarat tanda

Kita memakai batu persegi dengan sisi $a$, massa $M$, sudut kemiringan $\theta$, dan sudut datang $\beta$. Kombinasi

$$C=C_l\cos\theta-C_f\sin\theta$$

harus positif agar gaya hidrodinamik mempunyai komponen angkat ke atas. Kecepatan vertikal pada awal tumbukan adalah $v_z(0)=-v_x(0)\tan\beta<0$; kecepatan vertikal pada awal terbang bebas harus positif.

In [ ]:
import os
import numpy as np
import matplotlib

if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

G = 9.81                 # m/s^2
RHO_AIR = 1000.0        # kg/m^3
PARAMETER = {
    "massa": 0.120,              # kg
    "sisi": 0.100,               # m
    "theta": np.deg2rad(20.0),  # rad
    "beta": np.deg2rad(10.0),   # rad
    "C_l": 1.00,
    "C_f": 0.10,
    "v_x": 4.00,                # m/s
}

theta = PARAMETER["theta"]
beta = PARAMETER["beta"]
C_angkat = PARAMETER["C_l"] * np.cos(theta) - PARAMETER["C_f"] * np.sin(theta)
mu_efektif = (PARAMETER["C_l"] * np.sin(theta) + PARAMETER["C_f"] * np.cos(theta)) / C_angkat
v_z_masuk = -PARAMETER["v_x"] * np.tan(beta)

assert C_angkat > 0.0
assert mu_efektif > 0.0
assert v_z_masuk < 0.0
print(f"NumPy {np.__version__}; Matplotlib {matplotlib.__version__}")
print(f"C = {C_angkat:.6f}; mu = {mu_efektif:.6f}; v_z(0) = {v_z_masuk:.6f} m/s")

## 2. Fase terbang bebas

Jika batu meninggalkan air dengan $v_z(0)>0$, maka

$$X(t)=X(0)+v_x(0)t,\qquad Z(t)=v_z(0)t-\tfrac12gt^2.$$

Waktu hingga kembali ke ketinggian permukaan adalah $\tau_f=2v_z(0)/g$, dan panjang lompatannya $\lambda=v_x(0)\tau_f$.

In [ ]:
def fase_terbang_bebas(v_x, v_z_lepas, g=G, jumlah=501):
    """Kembalikan lintasan balistik dari permukaan ke permukaan."""
    assert v_x > 0.0 and v_z_lepas > 0.0 and g > 0.0
    tau_f = 2.0 * v_z_lepas / g
    t = np.linspace(0.0, tau_f, jumlah)
    x = v_x * t
    z = v_z_lepas * t - 0.5 * g * t**2
    return {"t": t, "x": x, "z": z, "tau_f": tau_f, "lambda": v_x * tau_f}

v_z_lepas = -v_z_masuk
terbang = fase_terbang_bebas(PARAMETER["v_x"], v_z_lepas)

assert np.isclose(terbang["z"][0], 0.0, atol=1e-14)
assert np.isclose(terbang["z"][-1], 0.0, atol=1e-14)
assert np.all(terbang["z"] >= -1e-14)
assert np.isclose(terbang["lambda"], PARAMETER["v_x"] * terbang["tau_f"])

fig, ax = plt.subplots(figsize=(8.2, 4.8), constrained_layout=True)
ax.plot(terbang["x"], 100.0 * terbang["z"], color="#0072B2", linewidth=2.5)
ax.axhline(0.0, color="#555555", linewidth=1.0)
ax.set(xlabel="jarak horizontal X (m)", ylabel="ketinggian Z (cm)", title="Lintasan satu fase terbang bebas")
ax.grid(alpha=0.25)
plt.show()
print(f"tau_f = {terbang['tau_f']:.4f} s; lambda = {terbang['lambda']:.4f} m")

**Deskripsi panjang Gambar 1:** Satu kurva biru berbentuk parabola dimulai pada permukaan air, naik ke puncak tunggal, lalu kembali ke permukaan. Sumbu mendatar menyatakan jarak horizontal dalam meter dan sumbu tegak menyatakan ketinggian dalam sentimeter. Kurva tidak pernah berada di bawah garis permukaan abu-abu selama fase terbang bebas.

## 3. Efek giroskopik dan konstanta integrasi

Dengan $\delta=\Omega_0(J_0-J_1)/J_1$, solusi kecepatan sudut pada fase terbang bebas adalah

$$\omega_{-y}(t)=\omega_{-y}(0)\cos(\delta t)+\omega_p(0)\sin(\delta t).$$

Karena $d\theta/dt=\omega_{-y}$, integral yang memenuhi kondisi awal adalah

$$\theta(t)=\theta(0)+\frac{\omega_{-y}(0)}{\delta}\sin(\delta t)+\frac{\omega_p(0)}{\delta}[1-\cos(\delta t)].$$

Suku $1-\cos(\delta t)$ penting: tanpanya, nilai pada $t=0$ tidak sama dengan $\theta(0)$. Untuk $\delta=0$, limitnya ialah $\theta(t)=\theta(0)+\omega_{-y}(0)t$.

In [ ]:
def solusi_giroskopik(t, theta_0, omega_y_0, omega_p_0, delta):
    """Solusi analitik yang kontinu juga pada delta=0."""
    t = np.asarray(t, dtype=float)
    if np.isclose(delta, 0.0, rtol=0.0, atol=1e-14):
        omega_y = np.full_like(t, omega_y_0)
        omega_p = np.full_like(t, omega_p_0)
        theta_t = theta_0 + omega_y_0 * t
    else:
        u = delta * t
        omega_y = omega_y_0 * np.cos(u) + omega_p_0 * np.sin(u)
        omega_p = omega_p_0 * np.cos(u) - omega_y_0 * np.sin(u)
        sin_per_delta = t * np.sinc(u / np.pi)
        one_minus_cos_per_delta = 2.0 * np.sin(0.5 * u) ** 2 / delta
        theta_t = theta_0 + omega_y_0 * sin_per_delta + omega_p_0 * one_minus_cos_per_delta
    return theta_t, omega_y, omega_p

t_giro = np.linspace(0.0, 0.70, 701)
theta_0 = np.deg2rad(20.0)
omega_y_0, omega_p_0 = 2.0, 0.8
Omega_0, J_0, J_1 = 40.0, 2.0, 1.0
delta = Omega_0 * (J_0 - J_1) / J_1
theta_giro, omega_y, omega_p = solusi_giroskopik(t_giro, theta_0, omega_y_0, omega_p_0, delta)
theta_tanpa, omega_y_tanpa, omega_p_tanpa = solusi_giroskopik(t_giro, theta_0, omega_y_0, omega_p_0, 0.0)

assert np.isclose(theta_giro[0], theta_0, atol=1e-14)
assert np.isclose(theta_tanpa[0], theta_0, atol=1e-14)
assert np.allclose(omega_y**2 + omega_p**2, omega_y_0**2 + omega_p_0**2, rtol=1e-13, atol=1e-13)
assert np.allclose(theta_tanpa, theta_0 + omega_y_0 * t_giro)

fig, ax = plt.subplots(figsize=(8.2, 4.8), constrained_layout=True)
ax.plot(t_giro, np.rad2deg(theta_giro), color="#0072B2", linewidth=2.3, label=f"berputar: delta = {delta:.0f} rad/s")
ax.plot(t_giro, np.rad2deg(theta_tanpa), color="#D55E00", linewidth=2.3, linestyle="--", label="tanpa putaran: delta = 0")
ax.axhline(np.rad2deg(theta_0), color="#555555", linewidth=1.0, label="theta(0)")
ax.set(xlabel="waktu (s)", ylabel="sudut theta (derajat)", title="Putaran membatasi perubahan sudut kemiringan")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

**Deskripsi panjang Gambar 2:** Kurva biru untuk batu yang berputar berosilasi sempit di sekitar sudut awal 20 derajat. Kurva jingga putus-putus untuk batu tanpa putaran meningkat hampir lurus dan segera menjauh dari sudut awal. Garis abu-abu horizontal menandai $\theta(0)$. Perbandingan ini memperlihatkan stabilisasi dalam model, bukan ukuran keberhasilan lemparan nyata.

## 4. Fase tumbukan untuk batu persegi

Sebelum batu terendam seluruhnya, pendekatan geometri $S_{im}=a|z|/\sin\theta$ mengubah persamaan vertikal menjadi

$$\ddot z+\omega_0^2z=-g,\qquad \omega_0^2=\frac{C\rho_wv_x(0)^2a}{2M\sin\theta},$$

dengan $z(0)=0$ dan $\dot z(0)=v_z(0)<0$. Solusinya adalah

$$z(t)=-\frac{g}{\omega_0^2}+\frac{g}{\omega_0^2}\cos(\omega_0t)+\frac{v_z(0)}{\omega_0}\sin(\omega_0t).$$

In [ ]:
def frekuensi_tumbukan_persegi(v_x, massa, sisi, theta, rho, C):
    assert v_x > 0.0 and massa > 0.0 and sisi > 0.0 and rho > 0.0
    assert 0.0 < theta < 0.5 * np.pi and C > 0.0
    return np.sqrt(C * rho * v_x**2 * sisi / (2.0 * massa * np.sin(theta)))

def keadaan_tumbukan(t, omega_0, v_z_0, g=G):
    """Posisi dan kecepatan vertikal pada model tumbukan konservatif."""
    u = omega_0 * np.asarray(t, dtype=float)
    z = -g / omega_0**2 + g * np.cos(u) / omega_0**2 + v_z_0 * np.sin(u) / omega_0
    v_z = -g * np.sin(u) / omega_0 + v_z_0 * np.cos(u)
    return z, v_z

def waktu_kunci_tumbukan(omega_0, v_z_0, g=G):
    """Waktu titik terdalam dan kemunculan kembali untuk v_z_0<0."""
    assert omega_0 > 0.0 and v_z_0 < 0.0
    fase = np.arctan(v_z_0 * omega_0 / g)
    return (np.pi + fase) / omega_0, 2.0 * (np.pi + fase) / omega_0

omega_0 = frekuensi_tumbukan_persegi(
    PARAMETER["v_x"], PARAMETER["massa"], PARAMETER["sisi"], theta, RHO_AIR, C_angkat
)
t_dalam, tau_c = waktu_kunci_tumbukan(omega_0, v_z_masuk)
t_tumbukan = np.linspace(0.0, tau_c, 901)
z_tumbukan, vz_tumbukan = keadaan_tumbukan(t_tumbukan, omega_0, v_z_masuk)
z_dalam, vz_dalam = keadaan_tumbukan(t_dalam, omega_0, v_z_masuk)
z_m_rumus = -G / omega_0**2 * (1.0 + np.sqrt(1.0 + omega_0**2 * v_z_masuk**2 / G**2))
batas_terendam = PARAMETER["sisi"] * np.sin(theta)

assert np.isclose(z_tumbukan[0], 0.0, atol=1e-14)
assert np.isclose(z_tumbukan[-1], 0.0, atol=2e-14)
assert np.isclose(vz_dalam, 0.0, atol=2e-14)
assert np.isclose(z_dalam, z_m_rumus, rtol=1e-13, atol=1e-14)
assert np.isclose(vz_tumbukan[-1], -v_z_masuk, rtol=1e-13, atol=1e-13)
assert abs(z_dalam) < batas_terendam

fig, ax1 = plt.subplots(figsize=(8.4, 5.0), constrained_layout=True)
ax1.plot(1000.0 * t_tumbukan, 1000.0 * z_tumbukan, color="#0072B2", linewidth=2.3, label="kedalaman z")
ax1.axhline(-1000.0 * batas_terendam, color="#CC79A7", linestyle="--", label="batas terendam seluruhnya")
ax1.axvline(1000.0 * t_dalam, color="#555555", linestyle=":", label="titik terdalam")
ax1.set(xlabel="waktu tumbukan (ms)", ylabel="z (mm)", title="Gerak vertikal selama satu tumbukan konservatif")
ax1.grid(alpha=0.25)
ax1.legend()
plt.show()
print(f"omega_0 = {omega_0:.3f} rad/s; tau_c = {1000*tau_c:.3f} ms; z_m = {1000*z_dalam:.3f} mm")

**Deskripsi panjang Gambar 3:** Kurva biru mulai pada kedalaman nol, turun ke satu titik minimum, lalu kembali ke nol. Garis vertikal abu-abu menandai titik terdalam. Garis magenta putus-putus, yang menandai kedalaman agar batu persegi terendam seluruhnya, berada jauh di bawah kurva. Dengan parameter demonstrasi ini, model memprediksi batu muncul kembali sebelum seluruh sisinya terendam.

## 5. Kecepatan kritis dan domain berlakunya rumus

Dengan $v_z(0)=-v_x(0)\tan\beta$, syarat agar kedalaman balik tetap lebih kecil daripada $a\sin\theta$ menghasilkan

$$v_x(0)^2>\frac{4Mg}{C\rho_wa^2}\left[1-\frac{2M\tan^2\beta}{a^3C\rho_w\sin\theta}\right]^{-1}.$$

Ruas kanan hanya bermakna untuk $C>0$ dan faktor di dalam kurung positif. Sel berikut sengaja memeriksa kedua syarat tanda sebelum menghitung akar.

In [ ]:
def kecepatan_kritis(massa, sisi, theta, beta, rho, C_l, C_f, g=G):
    C = C_l * np.cos(theta) - C_f * np.sin(theta)
    if not (massa > 0.0 and sisi > 0.0 and rho > 0.0 and 0.0 < theta < 0.5 * np.pi):
        return {"valid": False, "alasan": "parameter geometri atau bahan tidak valid"}
    faktor = 1.0 - 2.0 * massa * np.tan(beta) ** 2 / (sisi**3 * C * rho * np.sin(theta)) if C > 0.0 else np.nan
    if not (C > 0.0 and faktor > 0.0):
        return {"valid": False, "alasan": "C atau faktor kurung tidak positif", "C": C, "faktor": faktor}
    v_c = np.sqrt(4.0 * massa * g / (C * rho * sisi**2 * faktor))
    return {"valid": True, "C": C, "faktor": faktor, "v_c": v_c}

kritis = kecepatan_kritis(
    PARAMETER["massa"], PARAMETER["sisi"], theta, beta, RHO_AIR, PARAMETER["C_l"], PARAMETER["C_f"]
)
kasus_beta = []
for beta_derajat in (5.0, 10.0, 25.0, 45.0, 55.0):
    hasil = kecepatan_kritis(
        PARAMETER["massa"], PARAMETER["sisi"], theta, np.deg2rad(beta_derajat), RHO_AIR,
        PARAMETER["C_l"], PARAMETER["C_f"]
    )
    kasus_beta.append((beta_derajat, hasil))
    nilai = f"v_c = {hasil['v_c']:.4f} m/s" if hasil["valid"] else f"tidak berlaku ({hasil['alasan']})"
    print(f"beta = {beta_derajat:4.1f} derajat: {nilai}")

assert kritis["valid"]
assert PARAMETER["v_x"] > kritis["v_c"]
assert kasus_beta[-1][1]["valid"] is False

omega_pada_kritis = frekuensi_tumbukan_persegi(
    kritis["v_c"], PARAMETER["massa"], PARAMETER["sisi"], theta, RHO_AIR, C_angkat
)
vz_pada_kritis = -kritis["v_c"] * np.tan(beta)
t_dalam_kritis, _ = waktu_kunci_tumbukan(omega_pada_kritis, vz_pada_kritis)
z_pada_kritis, _ = keadaan_tumbukan(t_dalam_kritis, omega_pada_kritis, vz_pada_kritis)
assert np.isclose(abs(z_pada_kritis), PARAMETER["sisi"] * np.sin(theta), rtol=2e-13, atol=1e-14)
print(f"Kecepatan terpilih {PARAMETER['v_x']:.2f} m/s > v_c = {kritis['v_c']:.4f} m/s.")

## 6. Penyusutan panjang lompatan

Pendekatan kerja gesek bab memberi $v_{x,N}^2=v_{x,0}^2-2Ng\mu l$. Jika besar kecepatan vertikal antarlompatan dianggap tetap, maka

$$\lambda_N=\lambda_0\sqrt{1-N/N_c},\qquad N_c=\frac{v_{x,0}^2}{2g\mu l}.$$

Kita mengambil $l=v_{x,0}\tau_c$ dari simulasi tumbukan di atas. Ini adalah penutupan model sederhana, bukan hasil pengukuran panjang lintasan basah.

In [ ]:
def deret_lompatan(v_x_0, besar_v_z, mu, l, g=G):
    assert v_x_0 > 0.0 and besar_v_z > 0.0 and mu > 0.0 and l > 0.0
    kehilangan_v2 = 2.0 * g * mu * l
    N_c = v_x_0**2 / kehilangan_v2
    N = np.arange(0, int(np.floor(np.nextafter(N_c, -np.inf))) + 1)
    v_x_N = np.sqrt(np.maximum(0.0, v_x_0**2 - N * kehilangan_v2))
    lambda_N = 2.0 * besar_v_z * v_x_N / g
    return {"N": N, "v_x": v_x_N, "lambda": lambda_N, "N_c": N_c, "kehilangan_v2": kehilangan_v2}

l_basah = PARAMETER["v_x"] * tau_c
deret = deret_lompatan(PARAMETER["v_x"], abs(v_z_masuk), mu_efektif, l_basah)
V_c_gesek = np.sqrt(deret["kehilangan_v2"])
ambang_gabungan = max(kritis["v_c"], V_c_gesek)
lambda_0 = 2.0 * abs(v_z_masuk) * PARAMETER["v_x"] / G
lambda_rumus = lambda_0 * np.sqrt(1.0 - deret["N"] / deret["N_c"])

assert np.allclose(deret["lambda"], lambda_rumus, rtol=1e-14, atol=1e-14)
assert np.all(np.diff(deret["lambda"]) < 0.0)
assert np.isclose(deret["lambda"][0], lambda_0)
assert PARAMETER["v_x"] > ambang_gabungan

fig, ax = plt.subplots(figsize=(8.2, 4.8), constrained_layout=True)
ax.plot(deret["N"], deret["lambda"], color="#009E73", marker="o", linewidth=2.2, label="lambda_N")
ax.axvline(deret["N_c"], color="#D55E00", linestyle="--", label="N_c (ekstrapolasi model)")
ax.set(xlabel="jumlah tumbukan N", ylabel="panjang lompatan (m)", title="Panjang lompatan menyusut menurut model kerja gesek")
ax.grid(alpha=0.25)
ax.legend()
plt.show()
print(f"l = {l_basah:.5f} m; N_c = {deret['N_c']:.3f}; V_c = {V_c_gesek:.4f} m/s")
print(f"Ambang gabungan max(v_c, V_c) = {ambang_gabungan:.4f} m/s")

**Deskripsi panjang Gambar 4:** Titik-titik hijau yang dihubungkan garis menurun secara monoton dari panjang lompatan awal menuju nol ketika jumlah tumbukan bertambah. Garis jingga putus-putus menandai $N_c$, tempat rumus kontinu mencapai nol. Model tidak boleh diekstrapolasi melewati garis ini, dan pada praktiknya syarat kecepatan minimum dapat menghentikan deret lebih awal.

## 7. Pemeriksaan yang dapat dieksekusi

Sel terakhir mengulang perhitungan utama dan memeriksa kondisi awal, kekekalan besar kecepatan sudut, residual persamaan tumbukan, kedalaman kritis, determinisme, dan hukum penyusutan lompatan.

In [ ]:
# Terbang bebas: kondisi batas dan determinisme bit-demi-bit.
terbang_ulang = fase_terbang_bebas(PARAMETER["v_x"], v_z_lepas)
assert np.array_equal(terbang["t"], terbang_ulang["t"])
assert np.array_equal(terbang["x"], terbang_ulang["x"])
assert np.array_equal(terbang["z"], terbang_ulang["z"])
assert np.isclose(np.max(terbang["z"]), v_z_lepas**2 / (2.0 * G), rtol=2e-5)

# Giroskop: kondisi awal, norma, dan hubungan dtheta/dt = omega_-y.
theta_ulang, omega_y_ulang, omega_p_ulang = solusi_giroskopik(t_giro, theta_0, omega_y_0, omega_p_0, delta)
assert np.array_equal(theta_giro, theta_ulang)
assert np.array_equal(omega_y, omega_y_ulang) and np.array_equal(omega_p, omega_p_ulang)
assert np.isclose(theta_giro[0], theta_0, atol=1e-14)
turunan_theta = np.gradient(theta_giro, t_giro, edge_order=2)
assert np.max(np.abs(turunan_theta[2:-2] - omega_y[2:-2])) < 7e-4

# Tumbukan: residual analitik untuk z'' + omega_0^2 z = -g.
u = omega_0 * t_tumbukan
z_ddot = -G * np.cos(u) - v_z_masuk * omega_0 * np.sin(u)
residu_ode = z_ddot + omega_0**2 * z_tumbukan + G
assert np.max(np.abs(residu_ode)) < 6e-14
assert np.isclose(abs(z_dalam), abs(z_m_rumus), rtol=1e-13)
assert abs(z_dalam) < batas_terendam

# Ambang dan deret lompatan.
assert kritis["C"] > 0.0 and kritis["faktor"] > 0.0
assert np.isclose(abs(z_pada_kritis), batas_terendam, rtol=2e-13)
deret_ulang = deret_lompatan(PARAMETER["v_x"], abs(v_z_masuk), mu_efektif, l_basah)
assert np.array_equal(deret["N"], deret_ulang["N"])
assert np.array_equal(deret["lambda"], deret_ulang["lambda"])
assert np.all(1.0 - deret["N"] / deret["N_c"] > 0.0)

print("Semua pemeriksaan komputasi Bab 4 lulus.")
print("Hasil memvalidasi implementasi terhadap persamaan dan identitas analitik, bukan terhadap data eksperimen.")

## Kesimpulan

Notebook menunjukkan secara terpisah apa yang benar-benar mengikuti asumsi model. Fase terbang bebas menghasilkan lintasan parabola; konstanta integrasi yang benar menjaga $\theta(0)$; putaran normal membatasi perubahan kemiringan; model tumbukan persegi mempunyai titik balik yang dapat dihitung; dan kehilangan tetap pada $v_x^2$ menghasilkan hukum akar-kuadrat untuk panjang lompatan.

Batasnya sama pentingnya: $C$ dan faktor kurung harus positif, pendekatan luas terendam berhenti berlaku setelah batu terendam seluruhnya, model tumbukan mengabaikan disipasi ketika menghitung kemunculan kembali, dan tanpa data eksperimen kita belum menguji apakah parameter demonstrasi menggambarkan lemparan nyata.